# 09 — Final model-family benchmark

This notebook compares the six operational forecasting families used in the study:

1. **Baseline**
2. **Statistical**
3. **Machine learning**
4. **Deep learning**
5. **Advanced traditional**
6. **Robust hybrid**

It consumes the outputs generated by notebooks 04–08, assembles a common test-prediction panel, recomputes task-level metrics, ranks the model families, and creates the source table used for the manuscript.

Model and hyperparameter selection performed in notebooks 05–08 is not repeated here. Baseline and statistical representatives are selected from notebook 04 using validation RMSE only. Test data are used exclusively for the final comparison.

## Inputs and outputs

**Inputs**

- `results/baselines_statistical/07_all_predictions.csv`
- `results/machine_learning/14_selected_ml_test_predictions.csv`
- `results/deep_learning/15_selected_dl_test_predictions.csv`
- `results/advanced_traditional/predictions/02_selected_advanced_traditional_test_predictions.parquet`
- `results/robust_residual_hybrid/predictions/06_robust_test_predictions.parquet`

**Main outputs**

- `results/final_benchmark/predictions/01_family_selected_test_predictions.parquet`
- `results/final_benchmark/05_final_family_task_metrics.csv`
- `results/final_benchmark/07_aggregate_family_performance.csv`
- `results/final_benchmark/08_final_family_ranking.csv`
- `results/final_benchmark/09_table1_aggregate_predictive_performance.csv`
- `results/final_benchmark/31_rmse_r2_by_target_and_family.csv`

All paths are relative to the repository root.

In [ ]:
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

RESOLUTIONS = [4, 12, 20, 30, 60]
TARGETS = ["temperature", "relative_humidity"]
HORIZONS = [60, 120, 240, 480]

FAMILIES = ["BASELINE", "STATISTICAL", "MACHINE_LEARNING", "DEEP_LEARNING", "ADVANCED_TRADITIONAL", "HYBRID_ROBUST",]

FAMILY_LABELS = {
    "BASELINE": "Operational reference",
    "STATISTICAL": "Statistical",
    "MACHINE_LEARNING": "Machine learning",
    "DEEP_LEARNING": "Deep learning",
    "ADVANCED_TRADITIONAL": "Advanced traditional",
    "HYBRID_ROBUST": "Robust hybrid",
}

TASK_COLUMNS = ["resolution_minutes", "target", "horizon_minutes"]
KEY_COLUMNS = TASK_COLUMNS + ["origin_index"]

TRUTH_ATOL = 1e-3
TRUTH_RTOL = 1e-6
NRMSE_DDOF = 1

In [ ]:
def locate_repository_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (
            (candidate / "notebooks").exists()
            and (candidate / "results" / "baselines_statistical").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Repository root not found. Run notebooks 01–08 first and keep the standard folder structure."
    )


PROJECT_ROOT = locate_repository_root()
RESULTS_ROOT = PROJECT_ROOT / "results"
RESULTS_DIR = RESULTS_ROOT / "final_benchmark"
PREDICTION_DIR = RESULTS_DIR / "predictions"

if RESULTS_DIR.exists():
    shutil.rmtree(RESULTS_DIR)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

print("Repository structure located successfully.")
print("Output directory: results/final_benchmark/")

## 1. Load the selected predictions from notebooks 04–08

In [ ]:
INPUTS = {
    "baseline_statistical": RESULTS_ROOT / "baselines_statistical" / "07_all_predictions.csv",
    "machine_learning": RESULTS_ROOT / "machine_learning" / "14_selected_ml_test_predictions.csv",
    "deep_learning": RESULTS_ROOT / "deep_learning" / "15_selected_dl_test_predictions.csv",
    "advanced_traditional": RESULTS_ROOT / "advanced_traditional" / "predictions" / "02_selected_advanced_traditional_test_predictions.parquet",
    "robust_hybrid": RESULTS_ROOT / "robust_residual_hybrid" / "predictions" / "06_robust_test_predictions.parquet",
}

missing = [
    str(path.relative_to(PROJECT_ROOT))
    for path in INPUTS.values()
    if not path.exists()
]
if missing:
    raise FileNotFoundError(
        "Required upstream files were not found. Run notebooks 04–08 first:\n"
        + "\n".join(missing)
    )


def read_table(path):
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    return pd.read_csv(path)


raw = {name: read_table(path) for name, path in INPUTS.items()}

input_summary = pd.DataFrame([
    {
        "input": name,
        "relative_path": str(INPUTS[name].relative_to(PROJECT_ROOT)),
        "rows": len(frame),
        "columns": len(frame.columns),
    }
    for name, frame in raw.items()
])

display(input_summary)

In [ ]:
def require_columns(frame, required, name):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

COMMON_PREDICTION_COLUMNS = [ "resolution_minutes", "target", "horizon_minutes", "origin_index", "observed",]

require_columns(
    raw["baseline_statistical"],
    COMMON_PREDICTION_COLUMNS + ["predicted", "model", "family", "split"],
    "Notebook 04 predictions",
)
require_columns(
    raw["machine_learning"],
    COMMON_PREDICTION_COLUMNS + ["predicted", "model", "feature_set", "candidate_id"],
    "Notebook 05 selected test predictions",
)
require_columns(
    raw["deep_learning"],
    COMMON_PREDICTION_COLUMNS + ["predicted", "architecture", "candidate_id"],
    "Notebook 06 selected test predictions",
)
require_columns(
    raw["advanced_traditional"],
    COMMON_PREDICTION_COLUMNS + ["predicted", "family", "candidate_id"],
    "Notebook 07 selected test predictions",
)
require_columns(
    raw["robust_hybrid"],
    COMMON_PREDICTION_COLUMNS + ["robust_predicted", "operational_action", "operational_model"],
    "Notebook 08 robust test predictions",
)

print("Input schemas are compatible.")

## 2. Build one operational prediction set per family

Notebook 04 contains multiple baseline and statistical alternatives, so their representatives are selected per forecasting task using **validation RMSE**. The selected outputs from notebooks 05–08 are consumed directly.

In [ ]:
def prepare_predictions(frame):
    result = frame.copy()
    result["resolution_minutes"] = pd.to_numeric(result["resolution_minutes"]).astype(int)
    result["horizon_minutes"] = pd.to_numeric(result["horizon_minutes"]).astype(int)
    result["origin_index"] = pd.to_numeric(result["origin_index"]).astype(int)
    result["target"] = result["target"].astype(str).str.lower()

    result = result.loc[
        result["resolution_minutes"].isin(RESOLUTIONS)
        & result["target"].isin(TARGETS)
        & result["horizon_minutes"].isin(HORIZONS)
    ].copy()

    if "split" in result.columns:
        result["split"] = result["split"].astype(str).str.lower()

    return result


def validation_metrics(frame, group_columns):
    rows = []
    for keys, group in frame.groupby(group_columns, sort=False, observed=True):
        key_values = keys if isinstance(keys, tuple) else (keys,)
        y = group["observed"].to_numpy(dtype=float)
        p = group["predicted"].to_numpy(dtype=float)

        row = dict(zip(group_columns, key_values))
        row["validation_rmse"] = float(np.sqrt(mean_squared_error(y, p)))
        row["validation_r2"] = float(r2_score(y, p))
        rows.append(row)

    return pd.DataFrame(rows)


def select_notebook04_family(source, source_family, output_family):
    data = prepare_predictions(source)
    data = data.loc[
        data["family"].astype(str).str.lower().eq(source_family)
    ].copy()

    validation = data.loc[data["split"].eq("validation")].copy()
    scores = validation_metrics(
        validation,
        TASK_COLUMNS + ["model"],
    )

    winners = (
        scores.sort_values(
            TASK_COLUMNS + ["validation_rmse", "validation_r2", "model"],
            ascending=[True, True, True, True, False, True],
        )
        .groupby(TASK_COLUMNS, as_index=False, sort=False)
        .first()
    )

    test = data.loc[data["split"].eq("test")].merge(
        winners[TASK_COLUMNS + ["model"]],
        on=TASK_COLUMNS + ["model"],
        how="inner",
        validate="many_to_one",
    )

    test["family"] = output_family
    test["selected_configuration"] = test["model"].astype(str)

    return test


baseline_test = select_notebook04_family(
    raw["baseline_statistical"],
    "reference",
    "BASELINE",
)

statistical_test = select_notebook04_family(
    raw["baseline_statistical"],
    "statistical",
    "STATISTICAL",
)

In [ ]:
machine_learning_test = prepare_predictions(raw["machine_learning"])
if "split" in machine_learning_test.columns:
    machine_learning_test = machine_learning_test.loc[
        machine_learning_test["split"].eq("test")
    ].copy()
machine_learning_test["family"] = "MACHINE_LEARNING"
machine_learning_test["selected_configuration"] = (
    machine_learning_test["model"].astype(str)
    + ":"
    + machine_learning_test["feature_set"].astype(str)
    + ":candidate_"
    + machine_learning_test["candidate_id"].astype(str)
)


deep_learning_test = prepare_predictions(raw["deep_learning"])
if "split" in deep_learning_test.columns:
    deep_learning_test = deep_learning_test.loc[
        deep_learning_test["split"].eq("test")
    ].copy()
deep_learning_test["family"] = "DEEP_LEARNING"
deep_learning_test["selected_configuration"] = (
    deep_learning_test["architecture"].astype(str)
    + ":candidate_"
    + deep_learning_test["candidate_id"].astype(str)
)


advanced_test = prepare_predictions(raw["advanced_traditional"])
if "split" in advanced_test.columns:
    advanced_test = advanced_test.loc[
        advanced_test["split"].eq("test")
    ].copy()
advanced_test["selected_configuration"] = (
    advanced_test["family"].astype(str)
    + ":candidate_"
    + advanced_test["candidate_id"].astype(str)
)
advanced_test["family"] = "ADVANCED_TRADITIONAL"


robust_source = raw["robust_hybrid"].copy()
robust_source["predicted"] = robust_source["robust_predicted"]
robust_test = prepare_predictions(robust_source)
if "split" in robust_test.columns:
    robust_test = robust_test.loc[
        robust_test["split"].eq("test")
    ].copy()
robust_test["family"] = "HYBRID_ROBUST"
robust_test["selected_configuration"] = robust_test["operational_model"].astype(str)

## 3. Assemble a common test panel

All six families must refer to the same forecasting origins and observed target values. The advanced-traditional output is used as the common reference for this alignment.

In [ ]:
OUTPUT_COLUMNS = ( KEY_COLUMNS + ["observed", "predicted", "family", "selected_configuration"])

family_frames = {
    "BASELINE": baseline_test,
    "STATISTICAL": statistical_test,
    "MACHINE_LEARNING": machine_learning_test,
    "DEEP_LEARNING": deep_learning_test,
    "ADVANCED_TRADITIONAL": advanced_test,
    "HYBRID_ROBUST": robust_test,
}

prediction_frames = []

for family, frame in family_frames.items():
    require_columns(frame, OUTPUT_COLUMNS, family)

    if frame.duplicated(KEY_COLUMNS).any():
        raise ValueError(f"Duplicate prediction keys were found for {family}.")

    prepared = frame[OUTPUT_COLUMNS].copy()
    prepared["family"] = family
    prediction_frames.append(prepared)

final_predictions = pd.concat(prediction_frames, ignore_index=True)

canonical = final_predictions.loc[
    final_predictions["family"].eq("ADVANCED_TRADITIONAL")
].copy()

canonical_keys = (
    canonical[KEY_COLUMNS]
    .sort_values(KEY_COLUMNS)
    .reset_index(drop=True)
)

canonical_tasks = set(
    map(
        tuple,
        canonical[TASK_COLUMNS]
        .drop_duplicates()
        .sort_values(TASK_COLUMNS)
        .to_numpy(),
    )
)

for family in FAMILIES:
    subset = final_predictions.loc[
        final_predictions["family"].eq(family)
    ].copy()

    family_keys = (
        subset[KEY_COLUMNS]
        .sort_values(KEY_COLUMNS)
        .reset_index(drop=True)
    )

    if not family_keys.equals(canonical_keys):
        raise ValueError(
            f"Forecast-origin alignment differs for {family}."
        )

    family_tasks = set(
        map(
            tuple,
            subset[TASK_COLUMNS]
            .drop_duplicates()
            .sort_values(TASK_COLUMNS)
            .to_numpy(),
        )
    )
    if family_tasks != canonical_tasks:
        raise ValueError(
            f"Forecast-task coverage differs for {family}."
        )

    truth_check = canonical[
        KEY_COLUMNS + ["observed"]
    ].merge(
        subset[KEY_COLUMNS + ["observed"]],
        on=KEY_COLUMNS,
        how="inner",
        suffixes=("_reference", "_family"),
        validate="one_to_one",
    )

    if not np.allclose(
        truth_check["observed_reference"],
        truth_check["observed_family"],
        atol=TRUTH_ATOL,
        rtol=TRUTH_RTOL,
        equal_nan=False,
    ):
        raise ValueError(
            f"Observed target values differ for {family}."
        )

final_predictions = (
    final_predictions
    .drop(columns="observed")
    .merge(
        canonical[KEY_COLUMNS + ["observed"]],
        on=KEY_COLUMNS,
        how="left",
        validate="many_to_one",
    )
)

if not final_predictions[["observed", "predicted"]].apply(
    np.isfinite
).all().all():
    raise ValueError("Non-finite values were found in the final prediction panel.")

print("All model families share the same test origins and observed values.")

In [ ]:
robust_base_only = robust_test.loc[
    robust_test["operational_action"].astype(str).eq("BASE_ONLY"),
    KEY_COLUMNS + ["predicted"],
].rename(columns={"predicted": "hybrid_predicted"})

if len(robust_base_only):
    advanced_reference = advanced_test[
        KEY_COLUMNS + ["predicted"]
    ].rename(columns={"predicted": "advanced_predicted"})

    comparison = robust_base_only.merge(
        advanced_reference,
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )

    if not np.allclose(
        comparison["hybrid_predicted"],
        comparison["advanced_predicted"],
        atol=1e-12,
        rtol=0.0,
    ):
        raise ValueError(
            "BASE_ONLY hybrid predictions do not match the advanced-traditional forecasts."
        )

print("Hybrid BASE_ONLY forecasts are consistent with the advanced-traditional family.")

## 4. Recompute task-level test metrics

In [ ]:
metric_rows = []

for keys, group in final_predictions.groupby(
    ["family"] + TASK_COLUMNS,
    sort=False,
    observed=True,
):
    y = group["observed"].to_numpy(dtype=float)
    p = group["predicted"].to_numpy(dtype=float)

    if len(y) <= NRMSE_DDOF:
        raise ValueError(
            f"Insufficient observations for task {keys}."
        )

    target_sd = float(np.std(y, ddof=NRMSE_DDOF))
    rmse = float(np.sqrt(mean_squared_error(y, p)))

    metric_rows.append({
        "family": keys[0],
        "resolution_minutes": int(keys[1]),
        "target": keys[2],
        "horizon_minutes": int(keys[3]),
        "n_origins": len(group),
        "rmse": rmse,
        "nrmse": rmse / target_sd if target_sd > 0 else np.nan,
        "mae": float(mean_absolute_error(y, p)),
        "bias": float(np.mean(p - y)),
        "r2": float(r2_score(y, p)),
        "test_target_sd_ddof1": target_sd,
    })

task_metrics = pd.DataFrame(metric_rows)

if not task_metrics[["rmse", "nrmse", "mae", "r2"]].apply(
    np.isfinite
).all().all():
    raise ValueError("Non-finite metrics were found.")

task_metrics["rmse_rank"] = task_metrics.groupby(
    TASK_COLUMNS
)["rmse"].rank(method="average")

task_metrics["r2_rank"] = task_metrics.groupby(
    TASK_COLUMNS
)["r2"].rank(method="average", ascending=False)

task_metrics.to_csv(
    RESULTS_DIR / "05_final_family_task_metrics.csv",
    index=False,
)

task_summary = (
    task_metrics.groupby("family", as_index=False)
    .agg(
        tasks=("rmse", "size"),
        minimum_origins=("n_origins", "min"),
        maximum_origins=("n_origins", "max"),
    )
)

display(task_summary)

## 5. Aggregate performance and family ranking

In [ ]:
aggregate_performance = (
    task_metrics.groupby(["family", "target"], as_index=False)
    .agg(
        mean_rmse=("rmse", "mean"),
        mean_r2=("r2", "mean"),
        mean_nrmse=("nrmse", "mean"),
        median_nrmse=("nrmse", "median"),
        mean_rmse_rank=("rmse_rank", "mean"),
        tasks=("rmse", "size"),
    )
)

aggregate_performance["family_label"] = (
    aggregate_performance["family"].map(FAMILY_LABELS)
)

aggregate_performance.to_csv(
    RESULTS_DIR / "07_aggregate_family_performance.csv",
    index=False,
)

ranking = (
    task_metrics.groupby("family", as_index=False)
    .agg(
        average_rmse_rank=("rmse_rank", "mean"),
        median_rmse_rank=("rmse_rank", "median"),
        average_r2_rank=("r2_rank", "mean"),
        mean_nrmse=("nrmse", "mean"),
        median_nrmse=("nrmse", "median"),
        mean_r2=("r2", "mean"),
    )
    .sort_values(
        ["average_rmse_rank", "median_rmse_rank", "average_r2_rank", "family"]
    )
    .reset_index(drop=True)
)

ranking["family_label"] = ranking["family"].map(FAMILY_LABELS)

ranking.to_csv(
    RESULTS_DIR / "08_final_family_ranking.csv",
    index=False,
)

display(
    aggregate_performance.sort_values(
        ["target", "mean_rmse"]
    )
)
display(ranking)

## 6. Manuscript Table 1

In [ ]:
temperature_table = (
    aggregate_performance.loc[
        aggregate_performance["target"].eq("temperature"),
        ["family", "family_label", "mean_rmse", "mean_r2"],
    ]
    .rename(columns={
        "family_label": "Family",
        "mean_rmse": "Temperature RMSE (°C)",
        "mean_r2": "Temperature R²",
    })
)

humidity_table = (
    aggregate_performance.loc[
        aggregate_performance["target"].eq("relative_humidity"),
        ["family", "mean_rmse", "mean_r2"],
    ]
    .rename(columns={
        "mean_rmse": "Relative humidity RMSE (p.p.)",
        "mean_r2": "Relative humidity R²",
    })
)

table1 = temperature_table.merge(
    humidity_table,
    on="family",
    how="inner",
    validate="one_to_one",
)

family_order = ranking["family"].tolist()

table1["family"] = pd.Categorical(
    table1["family"],
    categories=family_order,
    ordered=True,
)

table1 = table1.sort_values("family").reset_index(drop=True)
table1["family"] = table1["family"].astype(str)

table1 = table1[
    [
        "family",
        "Family",
        "Temperature RMSE (°C)",
        "Temperature R²",
        "Relative humidity RMSE (p.p.)",
        "Relative humidity R²",
    ]
]

table1.to_csv(
    RESULTS_DIR / "09_table1_aggregate_predictive_performance.csv",
    index=False,
)

# Compatibility copy used by the manuscript workflow.
table1.to_csv(
    RESULTS_DIR / "31_rmse_r2_by_target_and_family.csv",
    index=False,
)

display(table1)

In [ ]:
manuscript_table = table1.drop(columns="family").copy()

manuscript_table.columns = pd.MultiIndex.from_tuples([
    ("", "Family"),
    ("Temperature", "RMSE (°C)"),
    ("Temperature", "R²"),
    ("Relative humidity", "RMSE (p.p.)"),
    ("Relative humidity", "R²"),
])


def emphasize_best_values(data):
    styles = pd.DataFrame(
        "",
        index=data.index,
        columns=data.columns,
    )

    for column in data.columns[1:]:
        values = pd.to_numeric(
            data[column],
            errors="raise",
        )

        best = (
            values.min()
            if "RMSE" in column[1]
            else values.max()
        )

        styles[column] = np.where(
            np.isclose(
                values.to_numpy(dtype=float),
                float(best),
                rtol=0.0,
                atol=1e-12,
            ),
            "font-weight: bold;",
            "",
        )

    return styles


formats = {
    ("", "Family"): "{}",
    ("Temperature", "RMSE (°C)"): "{:.4f}",
    ("Temperature", "R²"): "{:.4f}",
    ("Relative humidity", "RMSE (p.p.)"): "{:.4f}",
    ("Relative humidity", "R²"): "{:.4f}",
}

manuscript_style = (
    manuscript_table.style
    .hide(axis="index")
    .format(formats)
    .apply(emphasize_best_values, axis=None)
    .set_caption(
        "Table 1. Aggregate predictive performance of the model families "
        "for temperature and relative humidity."
    )
)

table1_html = RESULTS_DIR / "09_table1_manuscript_style.html"
table1_html.write_text(
    manuscript_style.to_html(),
    encoding="utf-8",
)

display(manuscript_style)

## 7. Save the common prediction panel

In [ ]:
final_predictions = (
    final_predictions
    .sort_values(["family"] + KEY_COLUMNS)
    .reset_index(drop=True)
)

final_predictions.to_parquet(
    PREDICTION_DIR / "01_family_selected_test_predictions.parquet",
    index=False,
)

output_summary = pd.DataFrame([
    {
        "output": "Family-selected test predictions",
        "relative_path": "results/final_benchmark/predictions/01_family_selected_test_predictions.parquet",
        "rows": len(final_predictions),
    },
    {
        "output": "Task-level metrics",
        "relative_path": "results/final_benchmark/05_final_family_task_metrics.csv",
        "rows": len(task_metrics),
    },
    {
        "output": "Aggregate family performance",
        "relative_path": "results/final_benchmark/07_aggregate_family_performance.csv",
        "rows": len(aggregate_performance),
    },
    {
        "output": "Final family ranking",
        "relative_path": "results/final_benchmark/08_final_family_ranking.csv",
        "rows": len(ranking),
    },
    {
        "output": "Manuscript Table 1",
        "relative_path": "results/final_benchmark/09_table1_aggregate_predictive_performance.csv",
        "rows": len(table1),
    },
])

display(output_summary)

print("Final model-family benchmark completed successfully.")
print("Results: results/final_benchmark/")

## Completion

The benchmark is complete when the notebook:

- loads the operational outputs from notebooks 04–08;
- aligns the same test origins and observed values across all six families;
- recomputes RMSE, NRMSE, MAE, bias, and R² from prediction-level test data;
- produces the common prediction panel and task-level metrics required by notebook 10;
- generates the aggregate family ranking and manuscript Table 1.

All reported values are calculated directly from the current pipeline outputs.